# NLP HW 02: Word Embeddings
## Submission By Sravanth Chowdary Potluri und5uv

#### *Assisted By Github Copilot and ChatGPT

Submission Deadline: __October 6, 2024, 11:59 PM__

A penalty will be applied for late submission. Please refer to the course policy for more detail.  

## Instructions

Please read the instructions carefully before you start working on the homework.

- Please follow instructions and printed out the results as required. Keep the printed results and your implementation for grading purpose.
    - The TAs will not run your code for grading purpose unless it is necessary. That means, you may lose some points if the printed results are not in the submitted file.
- Submission should be via Canvas.
    - If you use Google Colab for running the code, please download the file and submit it via Canvas once it's done.
    - Submission via a Google Colab link will be considered as an invalid submission.
- Please double check the submitted file once you upload it to Canvas.
    - Students should be responsible for checking whether they submit the right files.
    - Re-submission is not allowed once the deadline is passed.

Also, if you missed the class lectures, please study the course materials first before working on the homework. It may save you some time.

# Homework 02 Word Embeddings

### Goal

The **goal** of this homework is to provide an opportunity to build an end-to-end system.

Specifically, we are going to build a word embedding system, that can

1. Read and preprocess raw data
2. Use two different ways (latent semantic analysis and skip-gram) to learn word embeddings
3. Evaluate the quality of word embeddings using some intrinsic evaluation methods

### Submission

Your submission should only include this notebook file. Please keep **all the outputs** in your submission for grading. We will run the code only if we are not sure it is correct.

### Dependency

You will need the following package to finish this homework assignment

- [spaCy](https://pypi.org/project/spacy/)
- [fasttext](https://pypi.org/project/fasttext/)

### Hint

Search for the keyword `TODO` to find out which parts need your input

In [55]:
# Download the data from course webpage
import urllib.request
from os.path import isfile
if not isfile("embeddings/imdb-small.txt"):
    url = "https://yangfengji.net/uva-nlp-grad/data/embeddings.zip"
    print("Downloading ...")
    filename, headers = urllib.request.urlretrieve(url, filename="embeddings.zip")

    print("Decompressing the file ...")
    !unzip embeddings.zip

sents = open("embeddings/imdb-small.txt").read().split("\n")
print("Read {} sentences".format(len(sents)))

Decompressing the file ...
Archive:  embeddings.zip
   creating: embeddings/
  inflating: embeddings/imdb-small.txt  
  inflating: embeddings/word-pairs.txt  
Read 10000 sentences


## 1. Data Processing (5 points)

Data processing is an **essential** skill for NLP researchers. Unlike machine learning where researchers sometimes may want to use synthetic data to demonstrate the potential of their algorithms, NLP researchers need to deal with real-world data all the time. Unfortunately, this means that these data are noisy and often contain irregular patterns. Therefore, a reasonable data processing can alleviate the challenge of building NLP systems to some extent and may also help boost the performance of machine learning models.

Data processing for learning word embeddings includes two basic modules

- Tokenizing texts and replacing some special tokens
- Filtering low-frequency and building a vocab

### 1.1 Tokenization (2 points)

The following function *tokenize()* should include the following components

1. Load the raw text from the file named **imdb-small.txt**
2. Convert all characters into lower cases
3. Tokenize the raw text using `spaCy`
4. Remove all punctuation (as single tokens) and replace all numbers (as single tokens) with a special token `<num>`
5. Write the preprocessed text to the file named **imdb-small.txt.tokenized** and maintain the same format (one paragraph per line)

(The file names are pre-defined, please do not change them.)

In [56]:
# TODO: add necessary packages here
from spacy.lang.en import English

# Load the spaCy model
nlp = English()

def tokenize(infname="embeddings/imdb-small.txt"):
    # Open the input file
    with open(infname, 'r', encoding="utf-8") as infile:
        lines = infile.readlines()  # Read all lines from the input file
    
    # Open the output file
    outfname = open(infname + ".tokenized", "w", encoding="utf-8")
    
    # Process each line in the file
    for line in lines:
        line = line.lower()  # Convert to lowercase
        doc = nlp.tokenizer(line)  # Tokenize the line using spaCy

        processed_tokens = []
        for token in doc:
            if token.is_punct:
                # Skip punctuation
                continue
            elif token.like_num:
                # Replace numbers with <num>
                processed_tokens.append("<num>")
            else:
                # Keep the rest of the tokens
                processed_tokens.append(token.text)
        
        # Write the processed line to the output file
        outfname.write(" ".join(processed_tokens) + "\n")

    # Close the output file
    outfname.close()

### 1.2 Filtering (2 points)

The following function *token_filter()* should include the following components

1. Remove the words that appear in the data less than 5 times (word_frequency < 5)
2. Write the filtered data to the file named **imdb-small.txt.filtered** and maintain the same format (one sentence per line)
3. Return a Python list that contains all the words

In [57]:
# TODO: add necessary packages here
from collections import Counter

def token_filter(infname="embeddings/imdb-small.txt.tokenized", thresh=5):
    # Read the tokenized input file
    with open(infname, 'r', encoding="utf-8") as infile:
        lines = infile.readlines()  # Read all lines from the input file
    
    # Count the frequency of each word
    word_counter = Counter()
    for line in lines:
        words = line.strip().split()
        word_counter.update(words)

    # Filter out words that appear less than 'thresh' times
    vocab = [word for word, count in word_counter.items() if count >= thresh]
    vocab_set = set(vocab)  # Convert vocab list to set for fast lookup

    # Open the output file
    outfname = open(infname.replace(".tokenized", ".filtered"), 'w', encoding="utf-8")

    # Write the filtered sentences to the output file
    for line in lines:
        words = line.strip().split()
        filtered_words = [word for word in words if word in vocab_set]
        outfname.write(" ".join(filtered_words) + "\n")

    # Close the output file
    outfname.close()

    return vocab

### 1.3 Put all together (1 point)

The following code block will call the previous two functions to do data preprocessing.

This code block should include the following steps

- tokenization
- build the vocabulary with the variable name `vocab`
- print out the size of the vocabulary

In [58]:
tokenize()
vocab = token_filter()
print("The vocab size = {}".format(len(vocab)))

The vocab size = 18283


## 2. Word Embeddings (5 points)

In this section, you need to implement two different ways of constructing word embeddings: latent semantic analysis  and skipgram.

### 2.1 Latent semantic analysis (3 points)

The function of LSA should include the following components

- Construct the word-doc matrix using `CountVectorizer` with `tokenizer=lambda x : x.split()`, make sure in this matrix that each row represents one word and each column represents one document (sentence, to be accurate in this case)
- Use the `TruncatedSVD` from `sklearn.decomposition` to factorize the word-doc matrix
- Construct the word embedding matrix with dimensionality as $v \times k$, where $v$ is the vocab size and $k$ is the word embedding dimension

The LSA() function should return

- **embeddings**: A matrix with size $v\times k$ that contains all the word embeddings
- **vocab**: A Python dict with size $v$ that maps a word to the corresponding word index. Please pay attention to the mapping relation in vocab, which will be needed in the evaluation section.

In [59]:
# TODO: add necessary packages here
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
import numpy as np

def LSA(fname="embeddings/imdb-small.txt.filtered", dim=50):
    # Read the filtered sentences from the file
    sents = open(fname, 'r', encoding="utf-8").read().split("\n")
    sents = [sent for sent in sents if sent]  # Remove empty lines
    
    # Construct the word-document matrix using CountVectorizer
    vectorizer = CountVectorizer(tokenizer=lambda x: x.split())
    word_doc_matrix = vectorizer.fit_transform(sents)
    
    # Transpose the word-document matrix to make rows represent words
    word_doc_matrix = word_doc_matrix.T  # Now rows are words, columns are documents
    
    # Get the vocabulary (word to index mapping)
    vocab = {word: i for i, word in enumerate(vectorizer.get_feature_names_out())}

    # Apply TruncatedSVD to factorize the word-doc matrix
    svd = TruncatedSVD(n_components=dim)
    embeddings = svd.fit_transform(word_doc_matrix)

    # Ensure the embeddings matrix is v x k (words x embedding dim)
    embeddings = np.array(embeddings)

    return embeddings, vocab

### 2.2 Skip-gram model (2 points)

In this section, you do not have to implement the skip-gram model by yourself. An authentic implementation of skip-gram can be found in the Python package [fasttext](https://pypi.org/project/fasttext/), which you can install on the your local machine with the folllwing commandline or directly load the package if you are using Google Colab.
```python
pip install fasttext
```

In the following code, please use the `fasttext.train_unsupervised` function for the skipgram() implementation. For the `fasttext.train_unsupervised`, please use the following configurations

- `model='skipgram'`
- Context window size: `ws = 3`
- Word embedding dimension: `dim = 50`
- Number of negative examples: `neg = 5`

For all other parameters, use their default values.

Similar to the previous LSA(), Skipgram() should return

- **embeddings**: A matrix with size $v\times k$ that contains all the word embeddings
- **vocab**: A Python dict with size $v$ that maps an index to the corresponding word

To get the word embeddings and vocab from fasttext, you need to understand [some functions](https://pypi.org/project/fasttext/#api) provided by the `model` object in the fasttext.

In [60]:
# TODO: add necessary packages here
import fasttext
import numpy as np

def Skipgram(fname="embeddings/imdb-small.txt.filtered", ws=3, dim=50):
    # Train the skipgram model using fasttext
    model = fasttext.train_unsupervised(input=fname, model='skipgram', ws=ws, dim=dim, neg=5)
    
    # Get the vocabulary words and their indices
    words = model.get_words()
    # Create the vocab dictionary mapping index to word
    vocab = {word:i for i,word in enumerate(words)}
    
    # Create the embeddings matrix
    embeddings = np.array([model.get_word_vector(word) for word in words])
    
    return embeddings, vocab

### 2.3 Put all together

Run the following code blocks to get word embeddings from two different methods. It may take a couple of minutes to compute both embeddings.

In [61]:
embeddings_lsa, vocab_lsa = LSA()
embeddings_sg, vocab_sg = Skipgram()

Read 2M words
Number of words:  18284
Number of labels: 0
Progress: 100.0% words/sec/thread:  245278 lr:  0.000000 avg.loss:  2.376342 ETA:   0h 0m 0s


The following code will serve as the sanity check that `vocab_lsa` and `vocab_sg` contain the same words

In [62]:
lsa_word_set = set([item[0] for item in vocab_lsa.items()])
sg_word_set = set([item[0] for item in vocab_sg.items()])
sym_diff = lsa_word_set.symmetric_difference(sg_word_set)

if len(sym_diff) == 0:
    print("vocab_lsa and vocab_sg contain the same words!")
else:
    print("The word that only appear in one vocab: {}".format(sym_diff))

The word that only appear in one vocab: {'</s>'}


If the only word from the `symmetric_difference()` function is `</s>`, then your implementation should be fine. (`</s>` was added by `fasttext` automatically to the end of each text.)

## 3. Evaluation (5 points)

In this homework, we will only use intrinsic evaluation. Specifically, for a list of predefined word pairs with their similarity scores, the evaluation is to calculate the correlation between the predefined similarity scores and the cosine similarity scores based on word embeddings. The higher the correlation, the better the quality of word embeddings.

In [63]:
def load_wordpairs(fname = "embeddings/word-pairs.txt", vocab=None):
    records = {}
    with open(fname) as fin:
        for line in fin:
            items = line.strip().split(",")
            if (items[1] in vocab) and (items[2] in vocab): # make sure both words in the vocab
                records[(items[1],items[2])] = float(items[3])
    print("Load {} pairs of words for evaluation".format(len(records)))
    return records

### 3.1 Word similarity correlation (2 points)

The purpose of this section is to implement the correlation function that compares the predefined scores and the scores computed by cosine similarity. The code of the correlation function is almost done, and the only thing left is the code for computing cosine similarity.

In [64]:
# TODO: Add necessary packages here
import numpy as np
from scipy.stats import pearsonr

def cosine_similarity(vec1, vec2):
    # Compute cosine similarity between two vectors
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def correlation(records, embeddings, vocab):
    predefined_scores = []
    cossim_scores = []
    for (words, sim_score) in records.items():
        predefined_scores.append(sim_score)
        
        # Get the embeddings for both words
        word1_index = vocab[words[0]]
        word2_index = vocab[words[1]]
        
        # Get the embeddings for both words
        embedding1 = embeddings[word1_index]
        embedding2 = embeddings[word2_index]
        
        # Compute the cosine similarity between the two embeddings
        score = cosine_similarity(embedding1, embedding2)

        # Append the cosine similarity score
        cossim_scores.append(score)
    
    # Calculate the Pearson correlation between predefined scores and cosine similarity scores
    corr = pearsonr(predefined_scores, cossim_scores)
    return corr

Run the following code block to calculate the correlations between pre-defined similarity scores and the cosine similarity scores based on word embeddings

In [65]:
records_lsa = load_wordpairs(vocab=vocab_lsa)
corr_lsa = correlation(records_lsa, embeddings_lsa, vocab_lsa)
print("The correlation with the LSA embeddings = {} with p-value {}".format(corr_lsa[0], corr_lsa[1]))
records_sg = load_wordpairs(vocab=vocab_sg)
corr_sg = correlation(records_sg, embeddings_sg, vocab_sg)
print("The correlation with the SG embeddings = {} with p-value {}".format(corr_sg[0], corr_sg[1]))

if corr_lsa[0] > corr_sg[0]:
    print("LSA is better than Skip-gram")
elif corr_lsa[0] < corr_sg[0]:
    print("Skipgram is better than LSA")

Load 149 pairs of words for evaluation
The correlation with the LSA embeddings = 0.1583826791333579 with p-value 0.053703525512231946
Load 149 pairs of words for evaluation
The correlation with the SG embeddings = 0.3864192892006195 with p-value 1.129884013494028e-06
Skipgram is better than LSA


### 3.2 Analysis of context window size in Skipgram (3 points)

With the correlation function, we can analyze the effect of different context window sizes in the Skipgram model. Specifically, please call the previous implementation

- `Skipgram(fname, ws, dim=50)` with the context window size `ws` as 3, 6, 9, 12, 15
- For each context window size, calculate the correlation using the function `correlation(records, embeddings, vocab)`
- **Print out** the fives correlation scores in your final submission: one score per line with the following format
<center> ws\t correlation</center>

In [66]:
# TODO: add your code here

# TODO: add necessary packages here
def analyze_skipgram_context_window_size(fname="embeddings/imdb-small.txt.filtered", wordpairs_fname="embeddings/word-pairs.txt", dim=50):
    # Define different context window sizes
    context_window_sizes = [3, 6, 9, 12, 15]
    
    # Load the word pairs for evaluation
    vocab = None  # We will replace this later with the correct vocab from Skipgram model
    records = load_wordpairs(wordpairs_fname, vocab=vocab_sg)

    # Iterate over each window size, train Skipgram, and calculate correlation
    for ws in context_window_sizes:
        # Train the Skipgram model with the specified context window size
        embeddings, vocab = Skipgram(fname, ws, dim)

        # Calculate the correlation between predefined and cosine similarity scores
        corr = correlation(records, embeddings, vocab)

        # Print the window size and the corresponding correlation
        print(f"{ws}\t{corr[0]}")

# Call the analysis function
analyze_skipgram_context_window_size()

Load 149 pairs of words for evaluation


Read 2M words
Number of words:  18284
Number of labels: 0
Progress: 100.0% words/sec/thread:  221225 lr:  0.000000 avg.loss:  2.377614 ETA:   0h 0m 0s
Read 2M words
Number of words:  18284
Number of labels: 0


3	0.3733038091216463


Progress: 100.0% words/sec/thread:  139413 lr:  0.000000 avg.loss:  2.353193 ETA:   0h 0m 0s
Read 2M words
Number of words:  18284
Number of labels: 0


6	0.3528056584189979


...

Read 2M words
Number of words:  18284
Number of labels: 0
Progress: 100.0% words/sec/thread:   89505 lr:  0.000000 avg.loss:  2.260769 ETA:   0h 0m 0s


12	0.36404975355832037


Read 2M words
Number of words:  18284
Number of labels: 0
Progress:  99.5% words/sec/thread:   75730 lr:  0.000239 avg.loss:  2.210935 ETA:   0h 0m 0s

15	0.39705336801577573


Progress: 100.0% words/sec/thread:   75741 lr:  0.000000 avg.loss:  2.211705 ETA:   0h 0m 0s


Similar experiment can also be conducted on the parameter of negative examples `neg`, but it will not be included in this homework.